In [ ]:
# -*- coding: utf-8 -*-
"""
Compensação térmica pelo método de Park (1999) aplicada à parte real da impedância
+ Aplicação separada em treino e teste (sem vazamento de informação)
+ Classificação de falhas (RandomForestClassifier)
+ Split por temperatura (sem overlap)
+ RF regularizado (mas com capacidade suficiente)
Autor: Luiz Eduardo Abdala José (ajustado)
"""

import re, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)

# ========= PARÂMETROS =========
ARQ_BASE = "base-completo--.pkl"   # base com coluna 'falha' e 'temperatura_c'
REF_TEMP = 20                      # temperatura de referência para Park
FREQ_MIN_KHZ = 90
FREQ_MAX_KHZ = 120
SMOOTH_WIN = 5                     # suavização final (moving average)

# Parâmetros do Park (1999) – aplicados na PARTE REAL
PARK_MAX_SHIFT_FRAC = 0.25         # fração da largura de banda para τ_max
PARK_OVERLAP_MIN = 0.60           # fração mínima de overlap efetivo
PARK_SMOOTH_WIN = 5               # suavização dentro do Park
PARK_NSTEPS = 201                 # número de passos na busca de τ

# Random Forest (regularizado, mas ainda capaz)
RF_CLASSIF_PARAMS = dict(
    n_estimators= 100,
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    max_features=0.25,         # 25% das features por árvore
    bootstrap=True,
    max_samples=0.70,          # 70% das amostras por árvore
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# ========= FUNÇÕES AUXILIARES =========
def extract_freq_hz(col):
    """Extrai a frequência em Hz do nome da coluna 'f_xxxHz'."""
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None

def get_freq_columns(df, fmin_khz, fmax_khz):
    """Retorna colunas de frequências reais entre fmin e fmax (kHz), ordenadas."""
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None and fmin_khz <= f/1e3 <= fmax_khz:
            cols.append(c)
            freqs.append(f)
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.array(freqs)[order]

def shift_interp(x_row, fhz, tau_hz):
    """
    Aplica shift em frequência: f -> f + tau.
    Interpola de volta para a grade original fhz (Hz).
    """
    f_shift = fhz + float(tau_hz)
    return np.interp(fhz, f_shift, x_row, left=x_row[0], right=x_row[-1])

def moving_average(arr, win):
    """Média móvel simples, com padding nas bordas."""
    if win <= 1 or win % 2 == 0:
        return arr.copy()
    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode='edge')
    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode='valid')
    if len(smooth) > len(arr):
        smooth = smooth[:len(arr)]
    elif len(smooth) < len(arr):
        smooth = np.pad(smooth, (0, len(arr)-len(smooth)), mode='edge')
    return smooth

# ========= MÉTODO DE PARK (1999) – PARTE REAL =========
def park_compensate_single(x, y_ref, fhz,
                           max_shift_frac=PARK_MAX_SHIFT_FRAC,
                           overlap_min_frac=PARK_OVERLAP_MIN,
                           smooth_win=PARK_SMOOTH_WIN,
                           nsteps=PARK_NSTEPS):
    """
    Compensação térmica de Park aplicada à PARTE REAL da impedância.

    Busca τ (shift em frequência) e ΔS (offset em amplitude) que minimizam:
        Va(τ) = sum [ y_ref(f) - (x_shifted(f, τ) + ΔS) ]^2

    Onde:
        - x_shifted é a curva real deslocada em τ (Hz) via interpolação,
        - ΔS é a média da diferença y_ref - x_shifted no overlap,
        - τ é varrido em um intervalo simétrico em torno de 0.
    """
    n = len(x)
    fmin, fmax = fhz[0], fhz[-1]
    df_band = fmax - fmin
    tau_max = max_shift_frac * df_band

    tau_vals = np.linspace(-tau_max, tau_max, nsteps)
    best = (np.inf, 0.0, 0.0)  # (erro mínimo, tau_best, deltaS_best)

    for tau in tau_vals:
        x_shift = shift_interp(x, fhz, tau)
        xs = x_shift
        yr = y_ref

        # Overlap efetivo (checar tamanho mínimo)
        if len(xs) < int(overlap_min_frac * n):
            continue

        # ΔS é a média da diferença após o shift (como no Park)
        deltaS = float(np.mean(yr - xs))
        resid = yr - (xs + deltaS)
        Va = float(np.sum(resid * resid))

        if Va < best[0]:
            best = (Va, tau, deltaS)

    _, tau_best, dS_best = best

    # Reconstrói curva compensada com os melhores parâmetros
    yout = shift_interp(x, fhz, tau_best) + dS_best

    # Suavização final opcional
    if smooth_win > 1 and smooth_win % 2 == 1:
        yout = moving_average(yout, smooth_win)

    return yout, tau_best, dS_best

def park_batch(X, y_ref, fhz):
    """
    Aplica o método de Park (parte real) linha a linha:
    X: matriz (n_amostras, n_freq)
    y_ref: curva de referência (n_freq,)
    fhz: frequências em Hz (n_freq,)
    """
    n, m = X.shape
    Y = np.zeros_like(X)
    taus, deltas = [], []
    for i in range(n):
        yi, tau, dS = park_compensate_single(X[i], y_ref, fhz)
        Y[i] = yi
        taus.append(tau)
        deltas.append(dS)
    return Y, np.array(taus), np.array(deltas)

# ========= ETAPA 1 – CARREGAMENTO =========
print("🔹 Carregando base completa...")
df = pd.read_pickle(ARQ_BASE)

# Seleciona colunas de frequência (PARTE REAL) na faixa desejada
fcols, fhz = get_freq_columns(df, FREQ_MIN_KHZ, FREQ_MAX_KHZ)
fhz_khz = fhz / 1e3
print(f"Nº amostras: {len(df)} | Nº features (freq reais): {len(fcols)}")

# ========= ETAPA 2 – SPLIT POR TEMPERATURA (SEM OVERLAP) =========
temps_all = sorted(df["temperatura_c"].unique())
temps_train = temps_all[::2]  # pares de índices (não de valor)
temps_test  = temps_all[1::2]

print(f"\nTemperaturas de treino: {temps_train}")
print(f"Temperaturas de teste:  {temps_test}")

df_train_raw = df[df["temperatura_c"].isin(temps_train)].copy()
df_test_raw  = df[df["temperatura_c"].isin(temps_test)].copy()

print(f"Amostras treino (raw): {len(df_train_raw)} | teste (raw): {len(df_test_raw)}")

# ========= ETAPA 3 – REFERÊNCIA DE PARK (APENAS TREINO, SEM FALHA) =========
df_train_sem = df_train_raw[df_train_raw["falha"] == 0].copy()
pool_ref = df_train_sem.loc[np.isclose(df_train_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)

if len(pool_ref) == 0:
    print("⚠️ Nenhuma curva sem falha exata a 20°C no TREINO — usando mediana global sem falha do treino.")
    y_ref = np.median(df_train_sem[fcols].to_numpy(float), axis=0)
else:
    y_ref = np.median(pool_ref, axis=0)

print(f"Referência de Park calculada com {len(pool_ref)} curvas @ {REF_TEMP}°C (apenas TREINO, falha=0).")

# ========= ETAPA 4 – APLICAÇÃO DE PARK EM TREINO E TESTE =========
print("\n🔹 Aplicando compensação térmica (Park, 1999) no TREINO...")
X_train_raw = df_train_raw[fcols].to_numpy(float)
t0 = time.time()
Y_train, taus_train, deltas_train = park_batch(X_train_raw, y_ref, fhz)
print(f"✅ Park (treino) concluído em {time.time()-t0:.1f}s")

print("\n🔹 Aplicando compensação térmica (Park, 1999) no TESTE...")
X_test_raw = df_test_raw[fcols].to_numpy(float)
t0 = time.time()
Y_test, taus_test, deltas_test = park_batch(X_test_raw, y_ref, fhz)
print(f"✅ Park (teste) concluído em {time.time()-t0:.1f}s")

# Monta matrizes finais para classificação
X_train = Y_train
X_test  = Y_test
y_train = df_train_raw["falha"].to_numpy(int)
y_test  = df_test_raw["falha"].to_numpy(int)

print(f"\nAmostras após Park – treino: {len(X_train)} | teste: {len(X_test)}")

# ========= ETAPA 5 – CLASSIFICAÇÃO (FALHA) =========
print("\n🔹 Treinando RandomForestClassifier (RF regularizado, parte real)...")
clf = RandomForestClassifier(**RF_CLASSIF_PARAMS)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n== RESULTADOS RANDOM FOREST (Park, parte real, split térmico, sem leakage) ==")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

# ========= (OPCIONAL) TESTE DE DIAGNÓSTICO: TEMPERATURA COMO ALVO =========
print("\n================ TESTE DIAGNÓSTICO: TEMPERATURA COMO ALVO ================")
y_train_temp = df_train_raw["temperatura_c"].to_numpy(float)
y_test_temp  = df_test_raw["temperatura_c"].to_numpy(float)

clf_temp = RandomForestClassifier(**RF_CLASSIF_PARAMS)
clf_temp.fit(X_train, y_train_temp)
y_pred_temp = clf_temp.predict(X_test)

print("Matriz de confusão da TEMPERATURA (após Park, parte real):")
print(confusion_matrix(y_test_temp, y_pred_temp))
acc_temp = np.mean(y_pred_temp == y_test_temp)
print(f"Acurácia para temperatura: {acc_temp:.3f}")

print("\n✅ Execução concluída — Park aplicado apenas na PARTE REAL, sem vazamento de informação.")


In [ ]:
# ============================================================
# SUPER TESTE DE ROBUSTEZ: Leave-Temperatures-Out (LTO)
# ============================================================
import random

print("\n================ LEAVE-TEMPERATURES-OUT TEST ================")

# 1) Sorteia aleatoriamente 2 temperaturas para REMOVER do treino
temps_all_sorted = sorted(df["temperatura_c"].unique())
temps_remove = random.sample(list(temps_all_sorted), 2)

print(f"Temperaturas REMOVIDAS do treino: {temps_remove}")

# 2) Cria treino e teste novos
df_LTO_train = df[~df["temperatura_c"].isin(temps_remove)].copy()
df_LTO_test  = df[df["temperatura_c"].isin(temps_remove)].copy()

# 3) Referência de Park apenas com treino sem falha
df_LTO_train_sem = df_LTO_train[df_LTO_train["falha"] == 0]
pool_ref_LTO = df_LTO_train_sem.loc[
    np.isclose(df_LTO_train_sem["temperatura_c"], REF_TEMP), fcols
].to_numpy(float)

if len(pool_ref_LTO) == 0:
    y_ref_LTO = np.median(df_LTO_train_sem[fcols].to_numpy(float), axis=0)
else:
    y_ref_LTO = np.median(pool_ref_LTO, axis=0)

# 4) Aplica Park separado em treino e teste
X_LTO_train_raw = df_LTO_train[fcols].to_numpy(float)
X_LTO_test_raw  = df_LTO_test[fcols].to_numpy(float)

Y_LTO_train, _, _ = park_batch(X_LTO_train_raw, y_ref_LTO, fhz)
Y_LTO_test,  _, _ = park_batch(X_LTO_test_raw,  y_ref_LTO, fhz)

y_LTO_train = df_LTO_train["falha"].to_numpy(int)
y_LTO_test  = df_LTO_test["falha"].to_numpy(int)

# 5) Treina classificador
clf_LTO = RandomForestClassifier(**RF_CLASSIF_PARAMS)
clf_LTO.fit(Y_LTO_train, y_LTO_train)

# 6) Avalia nas temperaturas NUNCA vistas
y_LTO_pred = clf_LTO.predict(Y_LTO_test)

print("\n== RESULTADOS LTO (Temperaturas nunca vistas no treino) ==")
print(confusion_matrix(y_LTO_test, y_LTO_pred))
print(classification_report(y_LTO_test, y_LTO_pred, digits=3))


In [ ]:
# ===============================================================
# ===============================================================
#           🔥 TESTES COMPLETOS DE OVERFITTING EM SHM 🔥
# ===============================================================
# ===============================================================

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle
from sklearn.decomposition import PCA

# ---------------------------------------------------------------
# 1) TESTES BASELINE — CLASSIFICADORES FRACOS
# ---------------------------------------------------------------
print("\n================ TESTE 1 — BASELINES FRACOS ====================")

baselines = {
    "Logistic Regression (linear)": LogisticRegression(max_iter=500),
    "KNN (k=1)": KNeighborsClassifier(n_neighbors=1),
    "KNN (k=3)": KNeighborsClassifier(n_neighbors=3),
    "Naive Bayes": GaussianNB(),
    "SVM Linear": SVC(kernel='linear')
}

for name, model in baselines.items():
    try:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        acc = accuracy_score(y_test, pred)
        print(f"{name}: acurácia = {acc:.3f}")
    except Exception as e:
        print(f"{name}: ERRO ({e})")


# ---------------------------------------------------------------
# 2) PERMUTATION TEST — SHUFFLE DAS FEATURES
# ---------------------------------------------------------------
print("\n================ TESTE 2 — PERMUTATION TEST ====================")

X_train_perm = X_train.copy()
X_test_perm  = X_test.copy()

# embaralha independentemente cada feature
for j in range(X_train.shape[1]):
    X_train_perm[:, j] = shuffle(X_train_perm[:, j])
    X_test_perm[:, j]  = shuffle(X_test_perm[:, j])

clf_perm = RandomForestClassifier(**RF_CLASSIF_PARAMS)
clf_perm.fit(X_train_perm, y_train)
pred_perm = clf_perm.predict(X_test_perm)

acc_perm = accuracy_score(y_test, pred_perm)
print(f"Acurácia após PERMUTAR TODAS AS FEATURES: {acc_perm:.3f}")
print("(se > 0.50, há overfitting; se ~0.33, depende de classes balanceadas)")


# ---------------------------------------------------------------
# 3) TESTE DE ROBUSTEZ COM RUÍDO
# ---------------------------------------------------------------
print("\n================ TESTE 3 — ROBUSTEZ COM RUÍDO ====================")

sigma = 0.01 * np.std(X_train)    # 1% do desvio padrão da base
X_test_noisy = X_test + np.random.normal(0, sigma, X_test.shape)

clf_noise = RandomForestClassifier(**RF_CLASSIF_PARAMS)
clf_noise.fit(X_train, y_train)
pred_noise = clf_noise.predict(X_test_noisy)

acc_noise = accuracy_score(y_test, pred_noise)
print(f"Acurácia com ruído adicionado ao teste: {acc_noise:.3f}")


# ---------------------------------------------------------------
# 4) PCA 2D — VISUALIZAÇÃO DA SEPARAÇÃO ENTRE CLASSES
# ---------------------------------------------------------------
print("\n================ TESTE 4 — PCA 2D (visualização) ====================")

pca = PCA(n_components=2)
Z = pca.fit_transform(np.vstack([X_train, X_test]))
labels_all = np.hstack([y_train, y_test])

plt.figure(figsize=(7,5))
for c in np.unique(labels_all):
    plt.scatter(Z[labels_all==c, 0], Z[labels_all==c, 1], label=f"Classe {c}", s=30)
plt.title("PCA 2D das curvas após Park (parte real)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True)
plt.show()


# ---------------------------------------------------------------
# 5) LEAVE-FREQUENCIES-OUT (LFO)
# ---------------------------------------------------------------
print("\n================ TESTE 5 — LEAVE-FREQUENCIES-OUT (LFO) ====================")

n_feat = X_train.shape[1]
idx = np.random.choice(n_feat, size=n_feat//2, replace=False)   # metade das frequências

X_train_lfo = X_train[:, idx]
X_test_lfo  = X_test[:, idx]

clf_lfo = RandomForestClassifier(**RF_CLASSIF_PARAMS)
clf_lfo.fit(X_train_lfo, y_train)
pred_lfo = clf_lfo.predict(X_test_lfo)

acc_lfo = accuracy_score(y_test, pred_lfo)
print(f"Acurácia LFO (com apenas 50% das frequências): {acc_lfo:.3f}")
